# N048 · 链表合并与归并排序

**目标：** 复用节点完成有序合并并分析递归空间。

**先修：** N038, N045, N047。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 有序合并；断链；快慢分段；自底向上归并。

**配套讲解来源：** [同名逐章意见](../../comment/048_linked_list_merge_sort.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

数组排序你可以随便挑快排、归并、堆排，但**链表**没有 O(1) 的随机访问（想拿第 5 个元素只能从头走 5 步），快排那种"按下标挖坑交换"的玩法很吃亏。归并排序恰好只需要"找中点 + 顺序扫"，天生适合链表。这一章解决两个问题：

1. **合并两条有序链**（LC 21）：给定两条各自升序的链 a、b，把它们合成一条升序链。例子：输入 `1→3` 和 `2→4`，输出 `1→2→3→4`。为什么是它：每次比较两条链当前的开头，摘较小的接到结果尾部，1<2 先摘 1，然后 2<3 摘 2，然后 3<4 摘 3，最后 4 直接挂上。
2. **链表排序**（LC 148）：给定乱序链表头，返回排好序的链表头。例子：输入 `4→2→1→3→2`，输出 `1→2→2→3→4`。

本章实现的关键卖点是**复用原节点**：不 new 新节点、不搬数据，只靠改 next 指针完成排序。另外 “思路推导”部分 特意提醒：这是自顶向下的递归版，递归栈占 O(log n) 空间，**不能声称严格 O(1)**——这是一句防吹牛的诚实声明。

## 2. 基础知识：先读懂这些词

- **有序合并（merge）**：两条各自有序的链，只需反复比较两边的开头、摘较小者，就能线性合成一条整体有序的链。它是归并排序的发动机。
- **断链（切分）**：把一条链从中间剪成两条独立的短链，做法是"走到中点，把中点的 next 置为 None"。不断开的话，递归会以为还是一条链，逻辑就乱了。
- **快慢指针找中点**：链表不能 O(1) 跳到中点（“思路推导”部分 第一句），但 N047 的快慢指针可以线性找到：slow 走 1 步、fast 走 2 步，fast 到头时 slow 就停在切分点。本章让 fast 从 `head.next` 出发（比 slow 领先一步），使偶数长度时左半恰好分到 n/2 个节点。
- **分治/归并排序**：递归排序左半、递归排序右半，再合并两条有序短链。子问题每次严格变短，最终短到 0 或 1 个节点天然有序，直接返回。
- **稳定性（相等时优先取左链）**：合并比较时用 `<=`，两边值相等就摘左链的节点。这保证值相同的节点保持进入算法前的相对次序，即"稳定排序"。测试里专门有一条验证稳定性。
- **递归空间 O(log n)**：每层递归只用常数个局部变量，但递归深度约是 log₂n 层，系统调用栈要为每层留帧。十万个节点约 17 层，不是大数目，但它确实不是 O(1)——想真正 O(1) 空间得写"自底向上"的迭代版（见练习 1）。

## 3. 思路推导：从小例子开始

### 先攻合并：merge_two_lists

- **Step 1**：造 dummy，`tail` 指针表示"结果链的最后一个节点"，初始都指向 dummy。
- **Step 2**：只要 a、b 都还有节点：比较两边开头的值，较小的那个接到 `tail.next`，然后那条链的开头前进一步，`tail` 也前进一步。
- **Step 3**：某一边耗尽后，另一边剩下的整段本来就是有序的，直接 `tail.next = 剩下的那条`，一次接完。
- **Step 4**：返回 `dummy.next`。

**手算演示：** a=`2→4`，b=`1→3`。

| 轮次 | 比较 | 摘谁 | 结果链（挂好部分） | a 剩 | b 剩 |
|---|---|---|---|---|---|
| 1 | 2 vs 1 | 1 | 1 | 2→4 | 3 |
| 2 | 2 vs 3 | 2 | 1→2 | 4 | 3 |
| 3 | 4 vs 3 | 3 | 1→2→3 | 4 | （空） |
| 收尾 | b 空 | 整段接上 4 | 1→2→3→4 | — | — |

### 再攻排序：sort_list

- **Step 1**：递归出口——链表是空的或只有 1 个节点，天然有序，原样返回。
- **Step 2**：快慢指针找切分点（fast 从 head.next 出发），在 slow 之后断链，得到左链（从 head 到 slow）和右链 `second`。
- **Step 3**：递归排序两条短链。
- **Step 4**：把两条排好的链 merge，返回结果。

**手算演示（“运行示例”部分 的实例）：** 输入 `4→2→1→3→2`。表格里 notebook 直接给出 `('输入',[4,2,1,3,2])` 和 `('排序',[1,2,2,3,4])` 两行，中间过程如下。

- 第一刀（5 个节点，slow 停在第 3 个）：切成 `[4,2,1]` 和 `[3,2]`。
- 排 `[4,2,1]`：切成 `[4,2]` 和 `[1]`。排 `[4,2]` 再切成 `[4]`、`[2]`，单节点直接返回，merge 得 `[2,4]`；与 `[1]` 合并：先摘 1，再整段接上 `2→4`，得 `[1,2,4]`。
- 排 `[3,2]`：切成 `[3]`、`[2]`，merge 得 `[2,3]`。
- 最后合并 `[1,2,4]` 和 `[2,3]`：摘 1；比较 2 vs 2 相等，按约定**取左边的 2**；再比较 4 vs 2 摘 2、4 vs 3 摘 3；右链空，整段接上 4。结果 `[1,2,2,3,4]`，与表格一致。

## 4. 核心代码：merge_two_lists

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def merge_two_lists(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a else b
    return dummy.next
```

### 逐段读懂代码

“分段实现”部分 开头的 `ListNode`、`TreeNode`、`list_to_nodes`、`nodes_to_list` 与 N045 相同，不再重复。

### 1. `merge_two_lists` —— 有序合并

```python
def merge_two_lists(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a else b
    return dummy.next
```

- `dummy=tail=ListNode()`：链式赋值，dummy 和 tail 一开始是同一个空哨兵。dummy 负责钉住结果的开头，tail 负责跟踪结尾，分工之后互不干扰。
- `while a and b:`：两边都有货才需要比较；一边空了就进入收尾。
- `if a.val<=b.val:`：比较两条链各自的开头。注意是 `<=` 不是 `<`——相等时摘左链 a 的节点，这就是"相等时优先取左链节点，保留原始次序"（稳定性）的落点。
- `tail.next=a; a=a.next`：把较小的节点接到结果尾部，然后让那条链的开头前进一步。节点是**复用**的（原节点直接挂过来），没有复制数据。
- `tail=tail.next`：结果链的尾巴跟进一格。
- `tail.next=a if a else b`：条件表达式——a 还剩就接 a，否则接 b。一边必定已空，剩下的整段直接挂上，不用一个个摘。
- `return dummy.next`：真正的结果头。

### 2. `sort_list` —— 归并排序主函数

```python
def sort_list(head):
    if head is None or head.next is None:
        return head
    slow = head
    fast = head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    return merge_two_lists(sort_list(head), sort_list(second))
```

- `if head is None or head.next is None:`：递归出口——空链和单节点链天然有序，原样返回。这也是递归必然终止的原因：每个子链都严格更短，最终都落到这个出口。
- `slow=head; fast=head.next`：fast 领先一步出发（不是 N047 的同点出发）。这个偏移让偶数长度时 slow 精确停在左半最后一个节点上，左右两半长度更均衡。
- `while fast and fast.next:`：标准快慢走法，fast 及 fast.next 都存在才跨两步，防越界。
- `second=slow.next; slow.next=None`：**断链**——先把右半的头存下来，再把 slow 的 next 置 None 剪断。两行的顺序不能反：先剪的话右半的头就找不回来了。
- 最后一行：递归排序左链（head 开头）和右链（second 开头），再合并。两个递归调用的结果直接作为 merge 的实参，一行完成"分、治、合"的合。

**顺带盘点三个新手常见错误**，每个都藏在上面某一行里：

1. 合并时把 `a.val<=b.val` 写成 `a.val<b.val`：值相等时改成摘右链，相等节点的相对次序被交换，排序变成"不稳定"——对纯数字看不出差别，但对"值相同的两个不同对象"就错了，“自动测试”部分 第 4 条检查会当场抓住它。
2. 切分时先 `slow.next=None` 再取 `second`：右半的头在读出来之前就被 None 覆盖，`second` 变成 None，右半整段丢失，递归悄悄少排一半。两行的先后顺序是保命符。
3. 收尾写成 while 循环逐个摘：把 `tail.next=a if a else b` 展开成"只要 a 还有就摘一个"也能对，但多写几行还多几次指针赋值；一边剩的是一整段有序尾巴，一次接上最干净，也最不容易在展开时漏掉 `tail=tail.next`。

**再对照一次小数据，体会"分"的形状：** n=6 的链 `[5,4,3,2,1,0]`，第一刀 slow 停在第 3 个，切成 `[5,4,3]` 和 `[2,1,0]`；左边再切成 `[5,4]` 和 `[3]`，右边再切成 `[2,1]` 和 `[0]`；4 条子链各自最多再切一刀就到单节点。整棵递归树深度只有 3 层——这就是"每层链长减半、深度 log₂n"的具体样子。

## 5. 分段实现：按顺序运行

**本章接口：** `merge_two_lists(a, b)`、`sort_list(head)`

### ListNode

In [1]:
class ListNode:

    def __init__(self, val=0, next=None):
        self.val, self.next = (val, next)

### TreeNode

In [2]:
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

### list_to_nodes

In [3]:
def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

### nodes_to_list

In [4]:
def nodes_to_list(head):
    values, seen = ([], set())
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

### merge_two_lists

In [5]:
def merge_two_lists(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a else b
    return dummy.next

### sort_list

In [6]:
def sort_list(head):
    if head is None or head.next is None:
        return head
    slow = head
    fast = head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    return merge_two_lists(sort_list(head), sort_list(second))

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [7]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[4,2,1,3,2]
show_table(['阶段','链值'],[('输入',a),('排序',nodes_to_list(sort_list(list_to_nodes(a))))])

阶段,链值
输入,"[4, 2, 1, 3, 2]"
排序,"[1, 2, 2, 3, 4]"


## 7. 正确性、适用条件与复杂度

两个子链长度均严格减小；合并每次取两有序链头较小者，就是全局剩余最小值。链尾仅接剩余链一次，节点集合完整保持。

**代价：** O(n log n)时间、O(log n)递归辅助空间；链表重连不另建数据节点。

### 展开理解

**合并为什么对？** 两条链各自有序时，"全体剩余元素中的最小值"一定藏在两条链的开头之一（否则那个更小的元素前面还有比它大的，与有序矛盾）。所以每次摘下的开头较小者都是当前全局最小，一个个挂到结果尾部，结果自然整体有序。值相等时优先摘左链，左链节点本来就排在右链节点前面（相对次序），所以相等值的相对次序全程保持——稳定。最后剩余的一段整链接上，一段是有序的、且都大于已挂部分，直接拼没有问题。

**排序为什么对？** 用递归的标准论证：链长为 0 或 1 时直接返回，显然有序；链长为 n 时，两条子链的长度都严格小于 n（左半约 n/2、右半约 n/2），把"更短的链能排好"当作已经成立的事实，再由合并的正确性得到"两条有序短链合成一条有序整链"。子问题严格变短保证递归一定会到出口，不会无限分下去。“正确性与复杂度”部分 说"两个子链长度均严格减小"指的就是终止性。

**节点为什么不丢？** 切分时右半的头先存后剪，不会丢；合并时每个节点都是"被摘下、立刻挂上"，尾接只发生一次（`tail.next=a if a else b`）。整趟下来没有任何节点被 new 出来或被扔掉，节点集合原封不动，只是 next 的指向被重新安排——这正是"复用节点"。

**复杂度是多少？** 每一层递归要做的事总量是 O(n)（找中点扫一遍 + 合并扫一遍），递归深度约 log₂n 层（每层链长减半），所以时间是 O(n log n)。辅助空间方面：不建新数据节点，但递归栈有约 log₂n 层，每层常数个局部变量，所以是 O(log n)——“思路推导”部分 特别强调"不能声称严格 O(1)"。举个具体的数：十万个节点，log₂(10⁵) ≈ 17 层递归，每层总工作量十万级，总共约 170 万次操作，一瞬间算完；而冒泡排序要百亿级比较，根本不可同日而语。

**快慢指针的分段还保证了两半尽量均衡**：左半 ⌈n/2⌉、右半 ⌊n/2⌋，最多只差一个节点。均衡的分段是 O(n log n) 的前提——如果切分总是"1 个 vs n-1 个"（比如每次只切下一个节点），递归深度会退化成 n 层，总时间退化成 O(n²)，归并的优势就没了。fast 从 head.next 出发那一步偏移，正是在把切分点摆到正中间。

顺带一提：这个"分段必须均衡"的教训和数组快排完全同源——快排最怕也已排序的输入把 partition 切偏，归并排序则不管输入如何都从结构上强制对半，这也是它"稳定发挥"的根源。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试分两部分：

- `assert nodes_to_list(merge_two_lists(None,list_to_nodes([1,2])))==[1,2]`：**边界测试——一条链为空**。合并"空 + 有序"应该原样返回另一条。它考验的是 while 循环直接跳过、收尾表达式 `a if a else b` 正确选中非空那条。
- 后面是**随机化对拍 + 三重身份检查**（用固定种子 `Random(48)` 保证可复现）：对 n 从 0 到 39 的每个长度，生成值在 0 到 4 之间的随机链表（值域小，必然大量重复，专门考验稳定性），然后检查四件事：
  1. `assert p not in after`：排序结果里**没有任何节点出现两次**——既证明链没有意外成环（`nodes_to_list` 遇环会抛异常，这里提前拦），也证明没有节点被重复挂接。
  2. `[p.val for p in after]==sorted(a)`：结果的值序列与 Python 内置 `sorted` 完全一致——排序正确性对拍。
  3. `set(after)==set(before)`：排序前收集的全部节点对象与排序后的是**同一批对象**（集合按身份比较）——证明实现是原地的，没有偷偷新建节点、也没有弄丢节点。
  4. 对每个值 v：排序后值为 v 的节点序列，与排序前值为 v 的节点序列**逐一相等**（按对象身份）——这是**稳定性**的直接验证：相同值的节点保持了原来的相对次序。
- `print('N048: 所有本章断言通过')` 只是打一行确认信息。

这组测试比"排几个固定数组看输出"严格得多：它同时锁住了正确性、原地性、无环性和稳定性四项契约。

测试设计本身还有三个可学的门道：

1. **固定种子 `Random(48)`**：随机测试最怕"这次过了下次挂"，固定种子让每次运行生成完全相同的 40 组数据，失败可复现，这是工程上随机测试的标配做法。
2. **值域压到 0..4**：n 最多 39 而值只有 5 种，必然出现大量相等值，正好把"稳定性"这条最容易被忽略的契约逼出来考验——值互不相等的话，取左取右根本测不出差别。
3. **n 从 0 开始**：`range(40)` 的第一个值是 0，对应空链表（`list_to_nodes([])` 返回 None），`sort_list` 的第一个递归出口 `head is None` 就是被它测到的；长度 1、2 的小链则覆盖第二个出口和最小的一刀切分。

In [8]:
assert nodes_to_list(merge_two_lists(None, list_to_nodes([1, 2]))) == [1, 2]

from random import Random

rng = Random(48)

for n in range(40):
    a = [rng.randrange(5) for _ in range(n)]
    h = list_to_nodes(a)
    before = []
    p = h
    while p:
        before.append(p)
        p = p.next
    h = sort_list(h)
    after = []
    p = h
    while p:
        assert p not in after
        after.append(p)
        p = p.next
    assert [p.val for p in after] == sorted(a)
    assert set(after) == set(before)
    for value in set(a):
        assert [p for p in after if p.val == value] == [p for p in before if p.val == value]

print('N048: 所有本章断言通过')

N048: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 实现迭代自底向上链表归并。

**练习 2：** 说明递归版并非O(1)栈空间。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（迭代自底向上归并）**：方向是消掉递归。自底向上的思路：先两两合并长度 1 的段得到长度 2 的有序段，再合并长度 2 的得到长度 4 的，段长每轮翻倍直到覆盖整条链。实现要点是外层循环管段宽 `width=1,2,4,...`，内层每轮从剩下部分切出两段长度 width 的链、调用本章的 `merge_two_lists`，再用一个 dummy 收集结果。建议先用 `[4,2,1,3,2]` 手算第一轮：合并 (4,2)→(2,4)、合并 (1,3)→(1,3)、剩下 (2) 落单，第二轮链为 `2→4→1→3→2` 时合并长度 2 的段。验证时和本章递归版对拍随机输入。
- **练习 2（说明递归版并非 O(1) 栈空间）**：方向是数递归深度。每次调用 `sort_list` 在返回前要同时挂起两次子递归，调用栈最深等于递归树的高度 log₂n。建议手算 n=8 的递归树：1 层 1 个调用管 8 个节点、2 层 2 个调用各管 4 个、3 层 4 个调用各管 2 个、4 层 8 个调用直接返回，最深同时压 4 帧左右。可用 `sys.setrecursionlimit` 或在函数里传深度参数实测验证。结论写清楚：时间不变 O(n log n)，空间从 O(log n) 到迭代版的 O(1)，代价是代码复杂度上升。

两道练习的验证有一个共同的便宜办法：把本章的 `sort_list` 和 `list_to_nodes`、`nodes_to_list` 组合起来当"参考实现"，对同一批随机输入分别跑你的新实现和参考实现，断言两者输出的值序列逐个相等即可——不需要另外写暴力排序。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [9]:
# 完整实现：本单元格不依赖其他单元格。
class ListNode:

    def __init__(self, val=0, next=None):
        self.val, self.next = (val, next)

class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

def list_to_nodes(values):
    dummy = tail = ListNode()
    for value in values:
        tail.next = ListNode(value)
        tail = tail.next
    return dummy.next

def nodes_to_list(head):
    values, seen = ([], set())
    while head is not None:
        if id(head) in seen:
            raise ValueError('cycle in a supposedly finite list')
        seen.add(id(head))
        values.append(head.val)
        head = head.next
    return values

def merge_two_lists(a, b):
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a else b
    return dummy.next

def sort_list(head):
    if head is None or head.next is None:
        return head
    slow = head
    fast = head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    return merge_two_lists(sort_list(head), sort_list(second))

# 示例与回归测试
assert nodes_to_list(merge_two_lists(None, list_to_nodes([1, 2]))) == [1, 2]

from random import Random

rng = Random(48)

for n in range(40):
    a = [rng.randrange(5) for _ in range(n)]
    h = list_to_nodes(a)
    before = []
    p = h
    while p:
        before.append(p)
        p = p.next
    h = sort_list(h)
    after = []
    p = h
    while p:
        assert p not in after
        after.append(p)
        p = p.next
    assert [p.val for p in after] == sorted(a)
    assert set(after) == set(before)
    for value in set(a):
        assert [p for p in after if p.val == value] == [p for p in before if p.val == value]

print('N048: 所有本章断言通过')

N048: 所有本章断言通过


## 11. 代表题与迁移

- **21. Merge Two Sorted Lists（合并两个有序链表）**：本章 `merge_two_lists` 的原型题，练的是"dummy+tail 双指针摘头拼接 + 相等取左保稳定"。
- **148. Sort List（排序链表）**：本章 `sort_list` 的原型题，练的是"快慢指针断链分段 + 递归归并"，附带"为什么链表偏爱归并而不是快排"和"递归栈 O(log n)"这两层理解。

补充两个常见的迁移方向：一是"合并 K 条有序链"（LC 23），把 K 条链两两配对、反复调用本章的 `merge_two_lists`，就是链表版的归并思想推广；二是"插入排序排链表"（LC 147），它与本章互为反面教材——插入排序对链表是 O(n²)，归并对链表是 O(n log n)，对比之后你会更清楚"选算法要顺着数据结构的脾气"。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。